In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from scipy.special import expit
from tqdm import tqdm
import os
import random
import itertools

# ---------------- Configuration ----------------
CSV_FILE = '../Datasets/labeled_comments.csv'
MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 128
BATCH_SIZE = 16
NUM_TRAIN_EPOCHS = 5
LEARNING_RATE = 2e-5
TEST_SIZE = 0.2
RANDOM_SEED = 42
MINORITY_OVERSAMPLE_RATIO = 0.75
MINORITY_CLASS_WEIGHT_BOOST = 10.0

# Set seeds
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ---------------- Load Data ----------------
df = pd.read_csv(CSV_FILE)
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

# Normalize column names
df.columns = [c.replace('-', '_').replace(' ', '_') for c in df.columns]

# Labels
LABEL_COLUMNS = ['Sexism', 'Racism', 'Violence', 'Appearance', 'Ability', 'Non_offensive']
HATE_SPEECH_LABELS = [col for col in LABEL_COLUMNS if col != 'Non_offensive']

# Ensure label columns exist and are numeric
for col in LABEL_COLUMNS:
    if col not in df.columns:
        raise ValueError(f"Label column '{col}' not found in CSV")
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# Additional features
ADDITIONAL_TEXT_FEATURES = [f.replace('-', '_').replace(' ', '_') for f in [
    'Video Popularity', 'Comment Popularity', 'Gender of Sport_Female', 'Gender of Sport_Male',
    'Club_Arsenal', 'Club_Bayern München', 'Club_Chelsea', 'Club_Club América', 'Club_Corinthians',
    'Club_Juventus', 'Club_Manchester United', 'Club_Napoli', 'Club_Olympique Lyon',
    'Club_Paris FC', 'Club_Tottenham Hotspur', 'Club_Wolfsburg', 'Club Region_England',
    'Club Region_France', 'Club Region_Germany', 'Club Region_Italy', 'Club Region_Latin America',
    'Language_af', 'Language_ar', 'Language_bg', 'Language_bn', 'Language_ca', 'Language_cs',
    'Language_cy', 'Language_da', 'Language_de', 'Language_el', 'Language_en', 'Language_es',
    'Language_et', 'Language_fa', 'Language_fi', 'Language_fr', 'Language_he', 'Language_hi',
    'Language_hr', 'Language_hu', 'Language_id', 'Language_it', 'Language_ja', 'Language_ko',
    'Language_lt', 'Language_lv', 'Language_mk', 'Language_ml', 'Language_nl', 'Language_no',
    'Language_pl', 'Language_pt', 'Language_ro', 'Language_ru', 'Language_sk', 'Language_sl',
    'Language_so', 'Language_sq', 'Language_sv', 'Language_sw', 'Language_th', 'Language_tl',
    'Language_tr', 'Language_uk', 'Language_unknown', 'Language_ur', 'Language_vi',
    'Language_zh-cn', 'Language_zh-tw', 'Channel Structure_Seperate', 'Channel Structure_Unified',
    'Commenter Gender_male', 'Commenter Gender_unknown'
]]
ADDITIONAL_TEXT_FEATURES = [f for f in ADDITIONAL_TEXT_FEATURES if f in df.columns]

# ---------------- Stratified Split ----------------
def create_multilabel_stratified_split(df, label_columns, test_size=0.2, random_seed=42):
    np.random.seed(random_seed)
    train_indices, val_indices = set(), set()
    selected_indices = set()
    
    for col in label_columns:
        if col == 'Non_offensive':
            continue
        pos_idx = df[df[col] == 1].index.tolist()
        available = [i for i in pos_idx if i not in selected_indices]
        if len(available) > 0:
            n_val = max(1, min(len(available)//2, int(len(available)*test_size)))
            val_sel = np.random.choice(available, size=n_val, replace=False).tolist()
            val_indices.update(val_sel)
            selected_indices.update(val_sel)
            train_sel = [i for i in available if i not in val_sel]
            train_indices.update(train_sel)
            selected_indices.update(train_sel)
    
    unselected = [i for i in df.index if i not in selected_indices]
    rem_val = max(0, int(len(df)*test_size) - len(val_indices))
    if rem_val > 0 and len(unselected) > 0:
        add_val = np.random.choice(unselected, size=min(rem_val, len(unselected)), replace=False).tolist()
        val_indices.update(add_val)
        selected_indices.update(add_val)
        unselected = [i for i in unselected if i not in add_val]
    
    train_indices.update(unselected)
    selected_indices.update(unselected)
    
    return list(train_indices), list(val_indices)

train_idx, val_idx = create_multilabel_stratified_split(df, LABEL_COLUMNS, TEST_SIZE)
train_df = df.loc[train_idx].copy()
val_df = df.loc[val_idx].copy()

# ---------------- Dataset Class ----------------
class HateSpeechDataset(Dataset):
    def __init__(self, tokenizer, dataframe, label_columns, additional_features, max_length):
        self.tokenizer = tokenizer
        self.dataframe = dataframe.reset_index(drop=True)
        self.label_columns = label_columns
        self.additional_features = additional_features
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        comment = str(row['Comment']) if 'Comment' in row.index else ""
        feature_str = ""
        for f in self.additional_features:
            if f in row.index and pd.notna(row[f]):
                feature_str += f" {f.replace('_',' ').title()}: {row[f]}"
        text = f"{comment} [SEP] {feature_str.strip()}" if feature_str.strip() else comment
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        labels = torch.tensor([float(row[col]) if col in row.index else 0.0 for col in self.label_columns], dtype=torch.float)
        return {'input_ids': enc['input_ids'].flatten(), 'attention_mask': enc['attention_mask'].flatten(), 'labels': labels}

# ---------------- Model & Tokenizer ----------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(LABEL_COLUMNS), problem_type="multi_label_classification")
model.to(device)

train_dataset = HateSpeechDataset(tokenizer, train_df, LABEL_COLUMNS, ADDITIONAL_TEXT_FEATURES, MAX_LENGTH)
val_dataset = HateSpeechDataset(tokenizer, val_df, LABEL_COLUMNS, ADDITIONAL_TEXT_FEATURES, MAX_LENGTH)

# ---------------- Class Weights ----------------
pos_weights = []
for col in LABEL_COLUMNS:
    num_pos = train_df[col].sum()
    num_neg = len(train_df) - num_pos
    w = (num_neg / num_pos) if num_pos > 0 else 1.0
    if col in HATE_SPEECH_LABELS:
        w *= MINORITY_CLASS_WEIGHT_BOOST
    pos_weights.append(w)
pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float).to(device)

# ---------------- Robust Multilabel Balanced Sampler ----------------
class MultilabelBalancedBatchSampler(Sampler):
    def __init__(self, dataset, batch_size, label_columns, minority_oversample_ratio):
        self.dataset = dataset
        self.batch_size = batch_size
        self.label_columns = label_columns
        self.minority_oversample_ratio = minority_oversample_ratio
        self.minority_label_indices = [i for i, col in enumerate(label_columns) if col != 'Non_offensive']
        self.non_offensive_idx = label_columns.index('Non_offensive')

        self.label_to_indices = {i: [] for i in range(len(label_columns))}
        for i, row in enumerate(dataset.dataframe.itertuples(index=False)):
            labels = torch.tensor([getattr(row, col) for col in label_columns])
            for idx, val in enumerate(labels):
                if val == 1:
                    self.label_to_indices[idx].append(i)

        self.num_batches = len(dataset) // batch_size * 2

    def __iter__(self):
        label_iters = {idx: itertools.cycle(indices) for idx, indices in self.label_to_indices.items() if len(indices) > 0}
        num_minority = int(self.batch_size * self.minority_oversample_ratio)
        num_majority = self.batch_size - num_minority
        for _ in range(self.num_batches):
            batch = [next(label_iters[random.choice(self.minority_label_indices)]) for _ in range(num_minority)]
            batch += [next(label_iters[self.non_offensive_idx]) for _ in range(num_majority)]
            random.shuffle(batch)
            yield batch

    def __len__(self):
        return self.num_batches

train_sampler = MultilabelBalancedBatchSampler(train_dataset, BATCH_SIZE, LABEL_COLUMNS, MINORITY_OVERSAMPLE_RATIO)
train_dataloader = DataLoader(train_dataset, batch_sampler=train_sampler)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE*2, shuffle=False)

# ---------------- Training Setup ----------------
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights_tensor)

def get_scheduler(optimizer, warmup_steps, total_steps):
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: min((step+1)/max(1,warmup_steps), max(0.0, (total_steps-step)/max(1,total_steps-warmup_steps))))

scheduler = get_scheduler(optimizer, warmup_steps=min(500,len(train_dataloader)*NUM_TRAIN_EPOCHS//10), total_steps=len(train_dataloader)*NUM_TRAIN_EPOCHS)

# ---------------- Evaluation ----------------
def evaluate_model(model, dataloader, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += criterion(outputs.logits, labels).item()
            all_preds.append(expit(outputs.logits.cpu().numpy()))
            all_labels.append(labels.cpu().numpy())
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    preds = (all_preds > 0.5).astype(int)
    f1_micro = f1_score(all_labels, preds, average='micro', zero_division=0)
    f1_macro = f1_score(all_labels, preds, average='macro', zero_division=0)
    f1_per_class = f1_score(all_labels, preds, average=None, zero_division=0)
    try:
        roc_auc = roc_auc_score(all_labels, all_preds, average='macro')
    except ValueError:
        roc_auc = 0.0
    accuracy = accuracy_score(all_labels, preds)
    return {'loss': total_loss/len(dataloader), 'f1_micro': f1_micro, 'f1_macro': f1_macro, 'roc_auc_macro': roc_auc, 'accuracy': accuracy, 'f1_per_class': f1_per_class}

# ---------------- Training Loop ----------------
best_f1_macro = -1
best_model_state = None

for epoch in range(NUM_TRAIN_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_TRAIN_EPOCHS}")
    model.train()
    total_loss = 0
    for batch in tqdm(train_dataloader, desc=f"Training Epoch {epoch+1}"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = criterion(outputs.logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f"Average train loss: {total_loss/len(train_dataloader):.4f}")
    eval_results = evaluate_model(model, val_dataloader, device)
    print(f"Validation F1 Macro: {eval_results['f1_macro']:.4f}")
    if eval_results['f1_macro'] > best_f1_macro:
        best_f1_macro = eval_results['f1_macro']
        best_model_state = model.state_dict().copy()

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)

# Save model
model_save_path = "./fine_tuned_hate_speech_model"
os.makedirs(model_save_path, exist_ok=True)
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Model saved to {model_save_path}")


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Epoch 1/5


Training Epoch 1: 100%|██████████| 84/84 [1:27:15<00:00, 62.33s/it]


Average train loss: 60.5954


Evaluating: 100%|██████████| 6/6 [01:26<00:00, 14.48s/it]


...

Validation F1 Macro: 0.0149

Epoch 5/5


Training Epoch 5: 100%|██████████| 84/84 [56:24<00:00, 40.29s/it] 


Average train loss: 2.6643


Evaluating: 100%|██████████| 6/6 [00:35<00:00,  5.89s/it]


Validation F1 Macro: 0.0089
Model saved to ./fine_tuned_hate_speech_model
